### 2.3.4 具体的分割器
LangChain提供了许多不同类型的文档切分器：https://docs.langchain.com/oss/python/integrations/splitters

#### 2.3.4.1 CharacterTextSplitter：Split by character


类签名:

```python
CharacterTextSplitter(
  self,
  separator: str = '\n\n',
  is_separator_regex: bool = False,
  kwargs: Any = {}
)
```

签名和当前源码（`langchain-text-splitters` 1.1.2，`langchain_text_splitters/character.py`）一致，只有最后一个参数写法不同：

> ⚠️ **与当前源码不同**：签名里的 `kwargs: Any = {}` 应为 `**kwargs: Any`（任意个关键字参数）。`CharacterTextSplitter` 自己只定义了 `separator` 和 `is_separator_regex`，其余参数通过 `**kwargs` 原样交给父类 `TextSplitter.__init__`，也就是 2.3.3 表格里的 6 个参数。所以下面例子里的 `chunk_size`、`chunk_overlap` 都是父类的参数。


* chunk_size ：每个切块的最大长度，默认按字符数计，默认值为4000。只是“尽量不超过”：一个片段本身比它长时，会单独成为一块（见例 2）。
*  chunk_overlap ：相邻两个切块之间的最大重叠字符数量，默认值为200。为了保证段之间语义完整，可以设置每个块之间有一部分重叠。
* separator ：分割使用的分隔符，默认值为"\n\n"。
* length_function ：用于计算切块长度的方法。默认值是 Python 内置的 `len` 函数（在父类 TextSplitter 的 `__init__` 里设定），即按字符数计。


完整的参数：`CharacterTextSplitter` 自己定义 2 个，其余 6 个来自父类（含义见 2.3.3 的表格）：

| 参数 | 定义在 | 默认值 | 含义 |
|---|---|---|---|
| `separator` | `CharacterTextSplitter` | `"\n\n"` | 切片段用的分隔符。`CharacterTextSplitter` **只在分隔符处切**，片段再长也不会从内部切开。设为 `""` 时每个字符是一个片段 |
| `is_separator_regex` | `CharacterTextSplitter` | `False` | `separator` 是不是正则表达式。`False` 时源码先用 `re.escape` 转义，按普通字符串匹配；`True` 时按正则匹配，例如 `"[。！？]"` 可以同时按三种标点切（要配合 `keep_separator`，见最后的补充） |
| `chunk_size`、`chunk_overlap`、`length_function` | `TextSplitter` | 4000、200、`len` | 合并片段时使用，规则见 2.3.3 的 `_merge_splits` |
| `keep_separator` | `TextSplitter` | `False` | 切开后分隔符怎么处理。父类自己不用它，`CharacterTextSplitter` 在切片段时使用（见下面的源码和补充） |
| `strip_whitespace` | `TextSplitter` | `True` | 去掉每块首尾的空白（例 1 的块 1 就受它影响） |
| `add_start_index` | `TextSplitter` | `False` | 在 metadata 里记录块在原文中的起始位置，调用 `create_documents` 时生效 |

**返回值**：`split_text(text)` 返回 `list[str]`；`create_documents`、`split_documents` 继承自父类，返回 `list[Document]`（见 2.3.3）。官方文档的示例就是用 `create_documents` 把 metadata 带到每个块上。


**源码：`split_text` 分三步**

`CharacterTextSplitter` 的源码很短：`__init__` 保存 `separator`、`is_separator_regex`，把其余参数交给父类；`split_text` 分三步（注释是我加的）：

```python
def split_text(self, text: str) -> list[str]:
    # ① 分隔符转成正则：is_separator_regex=False 时先转义
    sep_pattern = (
        self._separator if self._is_separator_regex else re.escape(self._separator)
    )
    # ② 切成片段，keep_separator 决定分隔符的去留
    splits = _split_text_with_regex(
        text, sep_pattern, keep_separator=self._keep_separator
    )
    # ③ 决定合并时用什么拼接，再交给父类的 _merge_splits 合并
    #    分隔符是零宽断言（如 (?<=...)）时，切的时候没有删掉任何字符，不用拼回去
    lookaround_prefixes = ("(?=", "(?<!", "(?<=", "(?!")
    is_lookaround = self._is_separator_regex and any(
        self._separator.startswith(p) for p in lookaround_prefixes
    )
    #    分隔符已经留在片段里（keep_separator 为真）就用 "" 拼接，否则把 separator 原样拼回去
    merge_sep = ""
    if not (self._keep_separator or is_lookaround):
        merge_sep = self._separator
    return self._merge_splits(splits, merge_sep)
```

第 ② 步的 `_split_text_with_regex` 也定义在 `character.py` 里，做三件事：

* `separator` 不是空字符串：`keep_separator` 为假时直接 `re.split(separator, text)`，分隔符被删掉；为真时用带括号的正则 `re.split(f"({separator})", text)` 把分隔符也留在结果里，再接到后一个片段的开头（`True`、`"start"`）或前一个片段的末尾（`"end"`）；
* `separator` 是空字符串：`list(text)`，**每个字符一个片段**；
* 最后去掉空字符串。

第 ③ 步的 `_merge_splits` 是父类的方法（2.3.3 有源码和图）：依次把片段放进当前块，再放就超过 `chunk_size` 时输出这一块，然后保留末尾总长不超过 `chunk_overlap` 的片段，作为下一块的开头。

下面笔记的三个例子分别是：例 1 `separator=""`，例 2 `separator="。"`，例 3 `chunk_overlap=8`。最后两个补充例子看 `keep_separator` 和 `is_separator_regex`。

In [2]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。简单来说，原生的 LLM（如 GPT、Claude、DeepSeek 等）只负责根据输入的文本生成回答。但在实际开发中，AI 需要连接外部世界——比如查阅企业的私有数据库、搜索互联网、调用 API、或者记住多轮对话的上下文。LangChain 就充当了“大模型的操作系统或胶水”，把模型与外部数据、工具和工作流串联起来。
"""

# 3.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=50, # 每块大小
    chunk_overlap=5,# 块与块之间的重复字符数
    # length_function=len,
    separator=""   # 设置为空字符串时，表示禁用分隔符优先
)

# 4.分割文本
texts = splitter.split_text(text)

# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(repr(chunk))
    print("-" * 50)

块 1:长度：49
'LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发'
--------------------------------------------------
块 2:长度：50
'象，使开发者能够更容易地构建复杂的应用程序。简单来说，原生的 LLM（如 GPT、Claude、De'
--------------------------------------------------
块 3:长度：50
'de、DeepSeek 等）只负责根据输入的文本生成回答。但在实际开发中，AI 需要连接外部世界——'
--------------------------------------------------
块 4:长度：50
'部世界——比如查阅企业的私有数据库、搜索互联网、调用 API、或者记住多轮对话的上下文。LangCh'
--------------------------------------------------
块 5:长度：47
'angChain 就充当了“大模型的操作系统或胶水”，把模型与外部数据、工具和工作流串联起来。'
--------------------------------------------------


第一个块不足50个字符是因为，原本 `text` 开头存在换行符。而`CharacterTextSplitter`存在默认参数`strip_whitespace=True`会对每个 chunk 调用 .strip()，删除两端空白。删掉换行后剩下 49 个字符，不会再补一个字符进来。


例 1 中 `separator=""`，第 ② 步得到 `list(text)`：**每个字符一个片段**，换行、空格也各算一个。所以：

* 块 2～4 正好 50 个字符；块 1 少一个，是因为开头的换行被删掉了；块 5 是剩下的部分；
* 相邻两块正好重叠 5 个字符：块 2 开头的“象，使开发”就是块 1 的最后 5 个字；
* 切分点只看字数，不看内容：块 4 的结尾“LangCh”把单词 LangChain 切断了。

这就是 2.3.2 方法 2、3 说的“按固定字符数切 + 重叠窗口”。


In [3]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。简单来说，原生的 LLM（如 GPT、Claude、DeepSeek 等）只负责根据输入的文本生成回答。但在实际开发中，AI 需要连接外部世界——比如查阅企业的私有数据库、搜索互联网、调用 API、或者记住多轮对话的上下文。LangChain 就充当了“大模型的操作系统或胶水”，把模型与外部数据、工具和工作流串联起来。
"""

# 3.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=50, # 每块大小
    chunk_overlap=5,# 块与块之间的重复字符数
    # length_function=len,
    separator="。"   # 设置为空字符串时，表示禁用分隔符优先
)

# 4.分割文本
texts = splitter.split_text(text)

# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(repr(chunk))
    print("-" * 50)

Created a chunk of size 51, which is longer than the specified 50
Created a chunk of size 59, which is longer than the specified 50


块 1:长度：33
'LangChain 是一个用于开发由语言模型驱动的应用程序的框架的'
--------------------------------------------------
块 2:长度：31
'它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序'
--------------------------------------------------
块 3:长度：51
'简单来说，原生的 LLM（如 GPT、Claude、DeepSeek 等）只负责根据输入的文本生成回答'
--------------------------------------------------
块 4:长度：59
'但在实际开发中，AI 需要连接外部世界——比如查阅企业的私有数据库、搜索互联网、调用 API、或者记住多轮对话的上下文'
--------------------------------------------------
块 5:长度：48
'LangChain 就充当了“大模型的操作系统或胶水”，把模型与外部数据、工具和工作流串联起来。'
--------------------------------------------------


separator优先原则：当设置了 `separator` （如"。"），分割器只在分隔符处分割，再按 chunk_size 把片段合并成块；片段本身比 chunk_size 长时，也不会再从中间切开（例 2 的块 3、块 4）。这是为了避免在句子中间硬性切断。这种设计是为了：
1. 优先保持语义完整性（不切断句子）
2. 避免产生无意义的碎片（如半个单词/不完整句子）
3. 如果 chunk_size 比片段小，无法拆分片段，这个片段只能单独成为一块，导致 overlap失效。也就是说，片段足够短才能overlap
4. chunk_overlap仅在合并后的片段之间生效（如果 chunk_size 足够大）。如果没有合并的片段，则 overlap失效。

对照例 2 的输出：按“。”切出 6 个片段，长度是 34、31、51、59、47、1（第一个片段开头有换行，最后一个片段是文本末尾的换行）。

* 块 1、块 2 没有合并：34 + 1 + 31 = 66 > 50，合并时插回去的“。”也算 1 个字符；
* 块 3、块 4 超过 50，并打印了 `Created a chunk of size 51…` 警告：片段本身比 `chunk_size` 长，`CharacterTextSplitter` 不会再从中间切开。下一个要学的 `RecursiveCharacterTextSplitter` 遇到这种片段，会改用更细的分隔符继续切；
* 没有任何重叠：每块都只有一个片段（separator 优先原则第 4 点），而且每个片段都比 `chunk_overlap=5` 长（第 3 点）；
* 块 1～4 末尾的“。”没了：`keep_separator` 默认是 `False`，切开时分隔符被删掉，只有同一块内部合并时才把“。”拼回去。块 5 末尾有“。”，是因为它和最后那个换行片段合并时中间插入了“。”，换行又被 `strip_whitespace` 删掉了。想保留句号要设 `keep_separator="end"`（见最后的补充）。

说明：若必须禁用分隔符（`separator=""`，如处理无空格文本），块长基本等于 chunk_size，只有块首尾的空白被删掉时才会略小（如例 1 的块 1）；代价是句子、单词会被从中间切断。

In [4]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是第一段文本。这是第二段内容。最后一段结束。"

# 3.定义字符分割器
text_splitter = CharacterTextSplitter(
    chunk_size=20,
    chunk_overlap=8,
    separator="。",
    # keep_separator=True #chunk中是否保留切割符
)

# 4.分割文本
chunks = text_splitter.split_text(text)

# 5.打印结果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)

块 1:长度：15
这是第一段文本。这是第二段内容
--------------------------------------------------
块 2:长度：14
这是第二段内容。最后一段结束
--------------------------------------------------


例 3 有重叠：块 2 开头的“这是第二段内容”就是块 1 的末尾。按 `_merge_splits` 的规则走一遍：

1. 三个片段长 7、7、6。片段 1 + 片段 2 = 7 + 1 + 7 = 15；再放片段 3 是 15 + 1 + 6 = 22 > 20，输出块 1；
2. 从头部弹出片段 1，剩下片段 2：长 7 ≤ `chunk_overlap=8`，而且 7 + 1 + 6 = 14 ≤ 20，放得下片段 3，于是保留；
3. 块 2 = 片段 2 + 片段 3。

所以要产生重叠，上一块末尾的片段要同时满足两个条件：

* 总长 ≤ `chunk_overlap`（separator 优先原则第 3 点）；
* 加上下一个片段后不超过 `chunk_size`。上一块只有一个片段时，这一条一定不满足：它正是因为放不下下一个片段才单独成块的。所以单个片段的块不会产生重叠（第 4 点）。

改一改例 3 的参数，分别破坏这两个条件：

In [1]:
from langchain_text_splitters import CharacterTextSplitter

text = "这是第一段文本。这是第二段内容。最后一段结束。"

for chunk_size, chunk_overlap in [(20, 8), (20, 6), (14, 8)]:
    splitter = CharacterTextSplitter(
        separator="。", chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    print(f"chunk_size={chunk_size}, chunk_overlap={chunk_overlap} →", splitter.split_text(text))

chunk_size=20, chunk_overlap=8 → ['这是第一段文本。这是第二段内容', '这是第二段内容。最后一段结束']
chunk_size=20, chunk_overlap=6 → ['这是第一段文本。这是第二段内容', '最后一段结束']
chunk_size=14, chunk_overlap=8 → ['这是第一段文本', '这是第二段内容。最后一段结束']


* `(20, 8)`：例 3 本身，有重叠；
* `(20, 6)`：片段 2 长 7 > 6，被弹出，没有重叠；
* `(14, 8)`：片段 1 + 片段 2 = 15 > 14，块 1 只有片段 1，没有重叠。

#### 补充：`keep_separator`

例 3 里注释掉的 `keep_separator=True`，决定切开后“。”放在哪里。用例 3 的文本比较 4 种取值：

In [2]:
for keep in [False, True, "start", "end"]:
    splitter = CharacterTextSplitter(
        separator="。", chunk_size=20, chunk_overlap=8, keep_separator=keep
    )
    print(f"{keep!s:5} →", splitter.split_text(text))

False → ['这是第一段文本。这是第二段内容', '这是第二段内容。最后一段结束']
True  → ['这是第一段文本。这是第二段内容', '。这是第二段内容。最后一段结束。']
start → ['这是第一段文本。这是第二段内容', '。这是第二段内容。最后一段结束。']
end   → ['这是第一段文本。这是第二段内容。', '这是第二段内容。最后一段结束。']


* `False`（默认）：同一块内部合并时把“。”拼回去，但块末尾的“。”丢了；
* `True` 和 `"start"` 结果相同：“。”接在下一个片段的开头，块 2 以“。”开头，不符合中文习惯；
* `"end"`：“。”留在句末，切中文时最自然。分隔符已经留在片段里，所以第 ③ 步用 `""` 拼接，不会再插入一个“。”。

#### 补充：`is_separator_regex`

中文句子不一定以“。”结尾，还有“！”“？”。设 `is_separator_regex=True` 后 `separator` 按正则匹配，可以一次指定多种标点。但第 ③ 步有一个坑：`keep_separator=False` 时 `merge_sep` 就是 `separator` 本身，合并时拼回去的是**正则表达式的原文**：

In [ ]:
text = "七天无理由退货。退款多久到账？三个工作日内原路返回！电子发票在订单完成后开具。"

cases = {
    "1. 不当正则": dict(separator="[。！？]"),
    "2. 当正则，keep_separator=False": dict(separator="[。！？]", is_separator_regex=True),
    "3. 当正则，keep_separator='end'": dict(separator="[。！？]", is_separator_regex=True, keep_separator="end"),
    "4. 零宽断言：在句末标点后面切": dict(separator=r"(?<=[。！？])", is_separator_regex=True),
}
for name, kwargs in cases.items():
    splitter = CharacterTextSplitter(chunk_size=20, chunk_overlap=0, **kwargs)
    print(name, "\n   ", splitter.split_text(text))

1. 不当正则 
    ['七天无理由退货。退款多久到账？三个工作日内原路返回！电子发票在订单完成后开具。']
2. 当正则，keep_separator=False 
    ['七天无理由退货[。！？]退款多久到账', '三个工作日内原路返回', '电子发票在订单完成后开具']
3. 当正则，keep_separator='end' 
    ['七天无理由退货。退款多久到账？', '三个工作日内原路返回！', '电子发票在订单完成后开具。']
4. 零宽断言：在句末标点后面切 
    ['七天无理由退货。退款多久到账？', '三个工作日内原路返回！', '电子发票在订单完成后开具。']


* 1：`separator="[。！？]"` 这个字符串本身有 5 个字符：`[`、`。`、`！`、`？`、`]`。当正则时，方括号表示“匹配其中任意**一个**字符”，所以每次只匹配一个标点；不当正则时（`is_separator_regex=False`），它被 `re.escape` 转义成 `\[。！？\]`，只匹配这 5 个字符原样连在一起的字符串，原文里没有，所以整段成了一块。这一块 39 个字，超过了 `chunk_size` 却**没有警告**：`_merge_splits` 只在循环里输出块时检查长度，最后一块不检查；
* 2：按正则切开了，每次切掉一个标点。但块 1 里出现了 `[。！？]`：第 ③ 步的 `merge_sep = self._separator` 拼回去的是 `separator` 字符串本身（5 个字符，含两个方括号），而不是被切掉的那个标点。这 5 个字符也计入了块的长度：块 1 长 7 + 5 + 6 = 18；
* 3：`keep_separator="end"`，标点留在句末，结果正确。这次合并时为什么没有插入那 5 个字符，见下面的解释；
* 4：用零宽断言 `(?<=[。！？])` 切，不设 `keep_separator`，标点也留在句末，结果同样正确。原理见后面的“什么是零宽断言”。

**为什么例子 3 拼回去时没有插入 5 个字符**

关键在第 ③ 步决定 `merge_sep` 的那几行源码：

```python
merge_sep = ""
if not (self._keep_separator or is_lookaround):
    merge_sep = self._separator
return self._merge_splits(splits, merge_sep)
```

只有 `keep_separator` 为假、并且不是零宽断言时，`merge_sep` 才是 `separator` 字符串本身（例子 2 的 5 个字符）。例子 3 的 `keep_separator="end"` 为真，`merge_sep` 保持 `""`：合并时片段之间什么也不插入。

能用 `""` 拼接，是因为第 ② 步切片段时已经把标点留下了，而且留下的是原文里**实际匹配到的那个标点**。`keep_separator` 为真时，`_split_text_with_regex` 切分用的正则外面多了一对括号：`re.split(f"({separator})", text)`。下面直接调用这个函数看中间结果（它是下划线开头的内部函数，这里只用来观察）：

In [1]:
import re
from langchain_text_splitters.character import _split_text_with_regex

text = "七天无理由退货。退款多久到账？三个工作日内原路返回！电子发票在订单完成后开具。"
sep = "[。！？]"

print("re.split 不带括号：", re.split(sep, text))
print("re.split 带括号：  ", re.split(f"({sep})", text))
print()
print("例子 2 的片段（keep_separator=False）：", _split_text_with_regex(text, sep, keep_separator=False))
print("例子 3 的片段（keep_separator='end'）：", _split_text_with_regex(text, sep, keep_separator="end"))

re.split 不带括号： ['七天无理由退货', '退款多久到账', '三个工作日内原路返回', '电子发票在订单完成后开具', '']
re.split 带括号：   ['七天无理由退货', '。', '退款多久到账', '？', '三个工作日内原路返回', '！', '电子发票在订单完成后开具', '。', '']

例子 2 的片段（keep_separator=False）： ['七天无理由退货', '退款多久到账', '三个工作日内原路返回', '电子发票在订单完成后开具']
例子 3 的片段（keep_separator='end'）： ['七天无理由退货。', '退款多久到账？', '三个工作日内原路返回！', '电子发票在订单完成后开具。']


* 不带括号：匹配到的标点被删掉，结果里只剩句子；
* 带括号：括号在正则里叫**捕获组**。`re.split` 的规则是：正则里有捕获组时，匹配到的内容也放进结果列表。所以每个实际匹配到的标点（`。`、`？`、`！`）都保留了下来，夹在句子之间；
* `keep_separator="end"` 再把每个标点接到它前面那个片段的末尾（源码里是 `splits_[i] + splits_[i + 1]`），得到例子 3 的片段。最后的空字符串被去掉了；
* 两者对比：例子 2 的片段里已经没有标点，源码不知道每一处原来是哪个标点，合并时只能插回 `separator` 字符串，于是插进了 `[。！？]`；例子 3 的片段已经带着各自的标点，合并时用 `""` 直接相连就行，所以不会插入那 5 个字符。

**什么是零宽断言**

正则匹配到的通常是一段**字符**。例如 `[。！？]` 匹配到的是一个标点，占 1 个字符的宽度；`re.split` 在这里切开时，这个字符被当作分隔符删掉。

**零宽断言**（lookaround，也叫环视）匹配到的不是字符，而是两个字符之间的一个**位置**。它只检查这个位置的前面或后面是不是某个内容，被检查的内容不算进匹配结果。匹配结果的长度是 0，所以叫“零宽”。按零宽断言切分，就像在两个字符之间下刀，不会删掉任何字符。

Python 的正则有四种零宽断言（X 代表任意正则）：

| 写法 | 名称 | 匹配的位置 |
|---|---|---|
| `(?=X)` | 正向先行断言 | 后面紧跟着 X |
| `(?!X)` | 负向先行断言 | 后面不是 X |
| `(?<=X)` | 正向后行断言 | 前面紧挨着 X |
| `(?<!X)` | 负向后行断言 | 前面不是 X |

“先行”是看这个位置的后面，“后行”是看这个位置的前面。例子 4 的 `(?<=[。！？])` 是正向后行断言：匹配“前面紧挨着一个句末标点”的位置，也就是每个标点的后面。下面比较普通正则和零宽断言匹配到了什么、切出了什么：

In [2]:
text = "七天无理由退货。退款多久到账？三个工作日内原路返回！"

# finditer 列出每次匹配到的内容和位置 span=(开始, 结束)
print("普通正则 [。！？]：     ", [(m.group(), m.span()) for m in re.finditer(r"[。！？]", text)])
print("零宽断言 (?<=[。！？])：", [(m.group(), m.span()) for m in re.finditer(r"(?<=[。！？])", text)])
print()
print("按标点切：    ", re.split(r"[。！？]", text))
print("在标点之后切：", re.split(r"(?<=[。！？])", text))
print("在标点之前切：", re.split(r"(?=[。！？])", text))

普通正则 [。！？]：      [('。', (7, 8)), ('？', (14, 15)), ('！', (25, 26))]
零宽断言 (?<=[。！？])： [('', (8, 8)), ('', (15, 15)), ('', (26, 26))]

按标点切：     ['七天无理由退货', '退款多久到账', '三个工作日内原路返回', '']
在标点之后切： ['七天无理由退货。', '退款多久到账？', '三个工作日内原路返回！', '']
在标点之前切： ['七天无理由退货', '。退款多久到账', '？三个工作日内原路返回', '！']


* 匹配结果：`[。！？]` 匹配到标点本身，第一个“。”的 `span` 是 `(7, 8)`，长 1；`(?<=[。！？])` 匹配到的是空字符串 `''`，`span` 是 `(8, 8)`，长 0，这个位置就在“。”的后面；
* 按标点切：标点被删掉；
* 在标点之后切（`(?<=...)`，正向后行断言）：一个字符也没删，标点留在前一句的末尾，效果和 `keep_separator="end"` 一样；
* 在标点之前切（`(?=...)`，正向先行断言）：标点跑到了后一句的开头，效果和 `keep_separator="start"` 一样；
* 文本以标点结尾，前两种切法的结果最后多一个空字符串。`_split_text_with_regex` 最后会去掉空字符串，不影响 `CharacterTextSplitter` 的结果。

回到 `CharacterTextSplitter` 的源码：零宽断言切的时候没有删掉任何字符，合并时就不应该再插入分隔符。源码通过 `separator` 的开头判断它是不是零宽断言：

```python
lookaround_prefixes = ("(?=", "(?<!", "(?<=", "(?!")
is_lookaround = self._is_separator_regex and any(
    self._separator.startswith(p) for p in lookaround_prefixes
)
```

`is_separator_regex=True` 并且 `separator` 以这四种写法之一开头时，`is_lookaround` 为真，`merge_sep` 就是 `""`。所以例子 4 不用设 `keep_separator`，标点也能留在句末，合并时也不会插入正则原文。

所以正则分隔符要么配合 `keep_separator`，要么写成零宽断言。